In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET

KANJIVG_DIR = Path("data/kanji")

In [3]:
def get_components(character):
    filename = f"{ord(character):05x}.svg" # ord(character) converts the character to its Unicode number.  :05x converts that number to hexadecimal, padded to at least five digits.
    filepath = KANJIVG_DIR / filename

    tree = ET.parse(filepath)
    root = tree.getroot()

    namespace = "{http://kanjivg.tagaini.net}"

    components = []

    for element in root.iter():
        component = element.get(f"{namespace}element")

        if component and component != character:
            components.append(component)

    return components

In [4]:
for kanji in ["明", "林", "休", "顔"]:
    print(kanji, "→", get_components(kanji))

明 → ['日', '月']
林 → ['木', '木']
休 → ['亻', '木']
顔 → ['彦', '立', '厂', '厂', '彡', '頁', '貝', '目']


⚠️ A direct child group without `kvg:element` (880 of 6703 files, e.g. 三) is transparent: its own labelled children count as main components. Nested components below a labelled group are not returned.

In [5]:
SVG_GROUP_TAG = "{http://www.w3.org/2000/svg}g"
KVG_NAMESPACE = "{http://kanjivg.tagaini.net}"


def get_root_group(character):
    filepath = KANJIVG_DIR / f"{ord(character):05x}.svg"
    root = ET.parse(filepath).getroot()
    stroke_paths_group = next(
        g for g in root.iter(SVG_GROUP_TAG) if g.get("id", "").startswith("kvg:StrokePaths")
    )
    return next(g for g in stroke_paths_group if g.tag == SVG_GROUP_TAG)


def get_main_components(character):
    main_components = []
    pending_groups = [g for g in get_root_group(character) if g.tag == SVG_GROUP_TAG]
    while pending_groups:
        group = pending_groups.pop(0)
        element = group.get(KVG_NAMESPACE + "element")
        if element is not None:
            main_components.append(element)
        else:
            unlabelled_children = [g for g in group if g.tag == SVG_GROUP_TAG]
            pending_groups = unlabelled_children + pending_groups
    return main_components

In [8]:
for kanji in ["明", "東", "京", "都", "番", "冊"]:
    print(kanji, "→ main:", get_main_components(kanji), " all:", get_components(kanji))

明 → main: ['日', '月']  all: ['日', '月']
東 → main: ['木', '日', '木']  all: ['木', '日', '木', '丿']
京 → main: ['亠', '口', '小']  all: ['亠', '口', '小']
都 → main: ['者', '⻏']  all: ['者', '耂', '土', '日', '⻏']
番 → main: ['釆', '田']  all: ['釆', '丿', '米', '田']
冊 → main: ['冂', '廾']  all: ['冂', '丨', '廾', '丿', '十']


In [7]:
for kanji in ["明", "林", "休", "顔"]:
    filename = f"{ord(kanji):05x}.svg"
    filepath = KANJIVG_DIR / filename
    xml = filepath.read_text(encoding="utf-8")
    print(xml)

<?xml version="1.0" encoding="UTF-8"?>
<!--
Copyright (C) 2009/2010/2011 Ulrich Apel.
This work is distributed under the conditions of the Creative Commons
Attribution-Share Alike 3.0 Licence. This means you are free:
* to Share - to copy, distribute and transmit the work
* to Remix - to adapt the work

Under the following conditions:
* Attribution. You must attribute the work by stating your use of KanjiVG in
  your own copyright header and linking to KanjiVG's website
  (http://kanjivg.tagaini.net)
* Share Alike. If you alter, transform, or build upon this work, you may
  distribute the resulting work only under the same or similar license to this
  one.

See http://creativecommons.org/licenses/by-sa/3.0/ for more details.
-->
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.0//EN" "http://www.w3.org/TR/2001/REC-SVG-20010904/DTD/svg10.dtd" [
<!ATTLIST g
xmlns:kvg CDATA #FIXED "http://kanjivg.tagaini.net"
kvg:element CDATA #IMPLIED
kvg:variant CDATA #IMPLIED
kvg:partial CDATA #IMPLIED
kvg:orig